In [1]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time

In [6]:
import random
import math

In [35]:
random.seed(42)

In [52]:
tests = ['tsp_51_1', 'tsp_100_3', 'tsp_200_2', 'tsp_574_1', 'tsp_1889_1', 'tsp_33810_1']
thresholds = [(482, 430), (23433, 20800), (35985, 30000), (40000, 37600), (378069, 323000), (78478868, 67700000)]

In [60]:
def load_points(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n = int(lines[0].strip())
        points = list()
        for i in range(n):
            x, y = list(map(float, lines[1 + i].split()))
            points.append((x, y))

        return n, points

In [61]:
def check_tsp(n, points, order):
    used = [0] * n
    for i in order:
        if i >= n or i < 0:
            raise Exception("Not correct ordering")
            
        used[i] += 1
        if used[i] >= 2:
            raise Exception("Not correct ordering")
            
    result = 0
    for i in range(n):
        nxt = i + 1
        if nxt == n:
            nxt = 0

        pt1 = points[order[i]]
        pt2 = points[order[nxt]]
        
        result += math.dist(pt1, pt2)
        
    return result

In [62]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [63]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, points = load_points(test)
        start = time.time()
        
        if not use_file:
            order = method(n, points)
        else:
            order = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_tsp(n, points, order)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Напишем сначала наиболее простое решение, которое просто находит жадный порядок: всегда берет самое близкую вершину.

In [64]:
!g++ -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [65]:
def greedy_tsp(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        order = list(map(int, lines[0].split()))
        return order

In [66]:
test_method(greedy_tsp, "greedy", True)

Checking greedy
Execution time: 3.0335 seconds
Target function tsp_51_1: 506.363165362846
Passed tsp_51_1: 0
Execution time: 0.0196 seconds
Target function tsp_100_3: 25138.785452772318
Passed tsp_100_3: 0
Execution time: 0.0129 seconds
Target function tsp_200_2: 36226.22143814145
Passed tsp_200_2: 0
Execution time: 0.0149 seconds
Target function tsp_574_1: 47054.9474467063
Passed tsp_574_1: 0
Execution time: 0.0507 seconds
Target function tsp_1889_1: 391470.44549188914
Passed tsp_1889_1: 0
Execution time: 8.3191 seconds
Target function tsp_33810_1: 78478867.03022148
Passed tsp_33810_1: 1
Score: 3


Прошел только один тест, поэтому давайте это решение подтюним. А именно будем пробовать сделать reverse подотрезков в каком - то порядке, чтобы улучшить ответ и так делать, пока можем. 

In [77]:
!g++ -std=c++2a cpp_methods/greedy_local_opt.cpp -o tmp/greedy_local_opt

In [78]:
def greedy_local_opt_tsp(test_file): 
    os.system(f"./tmp/greedy_local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        order = list(map(int, lines[0].split()))
        return order

In [79]:
test_method(greedy_local_opt_tsp, "greedy_local_opt", True)

Checking greedy_local_opt
Execution time: 0.1506 seconds
Target function tsp_51_1: 440.15503849930053
Passed tsp_51_1: 1
Execution time: 0.0221 seconds
Target function tsp_100_3: 21746.189521377997
Passed tsp_100_3: 1
Execution time: 0.0501 seconds
Target function tsp_200_2: 31499.538298693467
Passed tsp_200_2: 1
Execution time: 0.4181 seconds
Target function tsp_574_1: 39538.653372590314
Passed tsp_574_1: 1
Execution time: 11.8531 seconds
Target function tsp_1889_1: 341010.65746338386
Passed tsp_1889_1: 1
Execution time: 68.8517 seconds
Target function tsp_33810_1: 77994569.32317169
Passed tsp_33810_1: 1
Score: 18


Прошли все простые пороги, 2-opt действительно неплохо работает в задаче TSP